<a href="https://colab.research.google.com/github/SimonHtet/Hotel/blob/main/exceltis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
## 1. Load
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Exceltis/Exeltis_Data_Engineer_Business_Case_Raw_Data.csv")
df.head()

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
0,O00001,9/13/2024 0:00,Mccoy Ltd,Hospitality,New Douglas,Portugal,44656,Nut Product 60,33,57.98,12.48,0.06,Snacks,Nuts,"Sims, Barber and Robinson",False,Sales Rep,Brenda Cook
1,O00002,1/27/2023 0:00,Porter and Sons,Distributor,Jorgefort,France,44372,Soft Drink Product 15,21,72.42,12.20,0.04,Beverages,Soft Drinks,"Rodriguez, Kelley and Myers",True,Phone,Kristin Burns
2,O00003,3/19/2023 0:00,Day Group,Distributor,Courtneystad,Portugal,45598,Nut Product 74,49,30.96,7.57,0.25,Snacks,Nuts,Greene-Bell,True,Sales Rep,Terri Mckay
3,O00004,9/6/2024 0:00,"Clark, Farrell and Sheppard",Distributor,New Molly,France,44921,Frozen Meal Product 32,90,58.39,38.22,0.30,Frozen,Frozen Meals,"Thomas, Harrison and Little",True,Online,Kevin Clark
4,O00005,1/15/2024 0:00,Hawkins and Sons,Distributor,Barnetthaven,France,45652,Cheese Product 73,6,27.35,35.41,0.15,Dairy,Cheese,Sharp-Cole,True,Online,Ethan Grant


In [3]:
## 2. Profile & identify data quality issues
# Check structure (dtypes), missing values, duplicates, and category placeholders before cleaning.

#Are the data types right?
df.info()
df.head()

#What's missing? Check nulls, category distributions, and duplicates
print(df.isnull().sum())
print()
print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print('Full duplicate rows:', df.duplicated().sum())
print('Duplicate order_id:', df['order_id'].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             2501 non-null   object 
 1   order_date           2501 non-null   object 
 2   customer_name        2501 non-null   object 
 3   client_segment       2501 non-null   object 
 4   client_city          2501 non-null   object 
 5   country              2501 non-null   object 
 6   client_singup_date   2501 non-null   object 
 7   product_name         2500 non-null   object 
 8   quantity             2501 non-null   int64  
 9   unit_price           2501 non-null   float64
 10  unit_cost            2500 non-null   float64
 11  discount_pct         2501 non-null   float64
 12  product_category     2500 non-null   object 
 13  product_subcategory  2500 non-null   object 
 14  supplier_name        2500 non-null   object 
 15  product_active       2500 non-null   o

In [4]:
df[['quantity','unit_price','unit_cost','discount_pct']].describe()

,quantity,unit_price,unit_cost,discount_pct
count,2501.000000,2501.000000,2500.000000,2501.000000
mean,50.004798,38.199592,19.320568,0.172367
std,28.827611,20.898183,11.705792,0.104649
min,-10.000000,2.010000,-5.000000,0.000000
25%,24.000000,19.900000,10.660000,0.080000
50%,50.000000,37.930000,17.770000,0.170000
75%,75.000000,55.880000,29.070000,0.260000
max,100.000000,74.990000,39.970000,1.500000


In [5]:
#some unit cost are negative,set to nan so it is treated as missing rather than a real value
df.loc[df['unit_cost'] <= 0, 'unit_cost'] = np.nan

In [6]:
print('Rows where cost >= price (negative margin):', (df['unit_cost'] >= df['unit_price']).sum())
print('Rows with inactive product sold :', (df['product_active'] == False).sum())



Rows where cost >= price (negative margin): 614
Rows with inactive product sold : 508


In [7]:
print(df['sales_channel'].value_counts())
print(df[['order_date','client_singup_date']].head())

sales_channel
Online       845
Phone        843
Sales Rep    813
Name: count, dtype: int64
       order_date client_singup_date
0  9/13/2024 0:00              44656
1  1/27/2023 0:00              44372
2  3/19/2023 0:00              45598
3   9/6/2024 0:00              44921
4  1/15/2024 0:00              45652


In [8]:
## 3. Clean & standardize
#signup dates are mixed:numbers and strings formats so I standardize it

def parse_mixed_dates(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, pd.Timestamp):
        return val
    s = str(val).strip()
    if s in ('','0','0.0'): #placeholder, not a real date
      return pd.NaT
    try:
        return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(float(s)))
    except (ValueError, TypeError):
        pass
    for fmt in ('%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d', '%Y/%d/%m'):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT

df['client_singup_date'] = df['client_singup_date'].apply(parse_mixed_dates)
print('nulls:', df['client_singup_date'].isna().sum())
print(df[['order_date','client_singup_date']].head())

nulls: 1
       order_date client_singup_date
0  9/13/2024 0:00         2022-04-05
1  1/27/2023 0:00         2021-06-25
2  3/19/2023 0:00         2024-11-02
3   9/6/2024 0:00         2022-12-26
4  1/15/2024 0:00         2024-12-26


In [9]:
print(df[df['customer_name'] == 'Richard, Simon and Madden'][['customer_name','client_singup_date']])

                  customer_name client_singup_date
68    Richard, Simon and Madden         2023-01-15
210   Richard, Simon and Madden         2023-01-15
593   Richard, Simon and Madden         2023-01-15
687   Richard, Simon and Madden         2023-01-15
951   Richard, Simon and Madden         2023-01-15
973   Richard, Simon and Madden         2023-01-15
1089  Richard, Simon and Madden         2023-01-15
1205  Richard, Simon and Madden         2023-01-15
1238  Richard, Simon and Madden         2023-01-15
1272  Richard, Simon and Madden         2023-01-15
1292  Richard, Simon and Madden         2023-01-15
1299  Richard, Simon and Madden         2023-01-15
1562  Richard, Simon and Madden         2023-01-15
1590  Richard, Simon and Madden         2023-01-15
1909  Richard, Simon and Madden         2023-01-15
2137  Richard, Simon and Madden         2023-01-15
2306  Richard, Simon and Madden         2023-01-15
2356  Richard, Simon and Madden         2023-01-15
2411  Richard, Simon and Madden

In [10]:
print(df['order_date'].iloc[80])

31-02-2024


In [11]:
df[df['order_date'] == '31-02-2024']

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
80,O00081,31-02-2024,Lee-Conway,Retail,Crystalshire,Italy,2023-02-16,Milk Product 4,72,25.94,20.96,0.21,Dairy,Milk,Mckee Group,True,Online,Amy Rodriguez


In [12]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce').astype('datetime64[ns]')
print('order_date Nat:', df['order_date'].isna().sum()) # see how many became NaT
print(df['order_date'].dtype)   # want: datetime64[ns]  (NOT object)

order_date Nat: 1
datetime64[ns]


In [13]:
df[df.duplicated(keep=False)] #finding duplicated rows

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
100,O00101,2024-11-01,Landry-Yang,Distributor,West Stephanie,France,2025-02-17,Water Product 65,56,9.24,4.12,0.25,Beverages,Water,"Jefferson, Barnes and Hammond",True,Sales Rep,Michelle Eaton
2500,O00101,2024-11-01,Landry-Yang,Distributor,West Stephanie,France,2025-02-17,Water Product 65,56,9.24,4.12,0.25,Beverages,Water,"Jefferson, Barnes and Hammond",True,Sales Rep,Michelle Eaton


In [14]:
df = df.drop_duplicates() #drop duplicates

In [15]:
print('duplicate order_ids:', df['order_id'].duplicated().sum())
print('rows now:',len(df))


duplicate order_ids: 0
rows now: 2500


In [16]:
df[df['discount_pct']>1]

,order_id,order_date,customer_name,client_segment,client_city,country,client_singup_date,product_name,quantity,unit_price,unit_cost,discount_pct,product_category,product_subcategory,supplier_name,product_active,sales_channel,sales_rep
15,O00016,2024-07-17,Duran-Rodriguez,Distributor,Osbornland,France,2022-02-16,Bread Product 75,61,42.4,27.21,1.5,Bakery,Bread,Nguyen-Horne,True,Phone,Jesus Reyes


In [17]:
# >100% is impossible so use np.nan (NOT pd.NA) so column stays float64
# pd.NA upcasts it to object and makes later arithmetic fragile.
df.loc[df['discount_pct'] > 1, 'discount_pct'] = np.nan
df['discount_pct'] = df['discount_pct'].astype('float64')   # guarantee numeric

print('discounts > 1 now:', (df['discount_pct'] > 1).sum())
print('discount_pct dtype:', df['discount_pct'].dtype)

discounts > 1 now: 0
discount_pct dtype: float64


In [18]:
print(df.groupby('product_name')[['supplier_name','product_active']].nunique().gt(1).sum())

supplier_name     1
product_active    1
dtype: int64


In [19]:
g=df.groupby('product_name')
for col in ['supplier_name', 'product_active']:
  bad = g[col].nunique()
  for p in bad[bad > 1].index:
    print(col, '|',p, '->', df.loc[df['product_name'] == p, col].value_counts().to_dict())

supplier_name | Ice Cream Product 22 -> {'Thomas Inc': 40, 'Smith Ltd': 30}
product_active | Ice Cream Product 22 -> {False: 40, True: 30}


In [20]:
df = df.copy()
df['country'] = df['country'].replace('0', 'Unknown')
df['client_segment'] = df['client_segment'].replace('0', 'Unknown')
df['customer_name'] = df['customer_name'].replace('0', 'Unknown')

print(df['country'].value_counts())
print(df['client_segment'].value_counts())
print(df['customer_name'].value_counts())

country
France      729
Portugal    675
Italy       588
Spain       507
Unknown       1
Name: count, dtype: int64
client_segment
Distributor    827
Retail         562
Hospitality    561
Supermarket    532
Unknown         18
Name: count, dtype: int64
customer_name
Clark, Farrell and Sheppard    54
Johnson, Glass and Foster      32
Petersen Inc                   31
Jordan, Russell and Smith      30
Graham, Knapp and Rivera       29
                               ..
Pearson, White and Wells       12
Stokes and Sons                12
Massey-Moreno                   8
King and Sons                   8
Unknown                         1
Name: count, Length: 120, dtype: int64


In [21]:
print(df.shape)
print(df['order_date'].dtype)
print((df['country']=='0').sum())

(2500, 18)
datetime64[ns]
0


In [22]:
print('row:', len(df))
print('dup order_id:', df['order_id'].duplicated().sum())
print('min quantity:', df['quantity'].min())
print('discount >1:', (df['discount_pct']>1).sum())
print("country '0':", (df['country'] == '0').sum(),"segment '0':",(df['client_segment']=='0').sum())

row: 2500
dup order_id: 0
min quantity: -10
discount >1: 0
country '0': 0 segment '0': 0


In [23]:
# normalize placeholders to NaN first
df['product_category']    = df['product_category'].replace('0', np.nan)
df['product_subcategory'] = df['product_subcategory'].replace('0', np.nan)

# derive product family (strip the trailing "Product N")
df['product_family'] = (df['product_name'].astype(str).str.replace(r'\s*Product\s+\d+$', '',regex=True).str.strip())

#fix both blanks and contradictions using the most common value per family
modal = lambda s: (s.dropna().mode().iloc[0] if len(s.dropna().mode()) else np.nan)

df['product_category'] =df['product_family'].map(df.groupby('product_family')['product_category'].agg(modal))
df['product_subcategory'] =df['product_family'].map(df.groupby('product_family')['product_subcategory'].agg(modal))

 # client_city placeholder fix (kept from before)
df['client_city'] = df['client_city'].str.strip().replace('0', 'Unknown')

 # verify Ice Cream is now consistent
print(df[df['product_name'].str.contains('Ice Cream', na=False)][['product_name','product_category','product_subcategory']].drop_duplicates())

            product_name product_category product_subcategory
29  Ice Cream Product 22           Frozen           Ice Cream
58  Ice Cream Product 43           Frozen           Ice Cream
77  Ice Cream Product 51           Frozen           Ice Cream


In [24]:
# === 4. Quarantine structurally-invalid rows ==
# Note: rows with valid revenue but MISSING cost are kept (cost/profit null) — documented decision.
reject_mask = (
      (df['quantity'] <= 0)
      | df['unit_price'].isna() | (df['unit_price'] <= 0)
      | df['product_name'].isna()
      | df['product_category'].isna()
      | df['order_date'].isna()
  )
rejects = df[reject_mask].copy()      # keep for export in cell 30
df = df[~reject_mask].copy()

print('clean rows:', len(df))
print('quarantined:', len(rejects))
print(rejects[['order_id','quantity','unit_price','product_name','product_category','order_date']])

clean rows: 2497
quarantined: 3
   order_id  quantity  unit_price     product_name product_category order_date
22   O00023       -10       26.87  Milk Product 61            Dairy 2023-09-16
35   O00036        57        7.87              NaN              NaN 2024-03-18
80   O00081        72       25.94   Milk Product 4            Dairy        NaT


In [25]:
print(df['order_date'].dtype)
print(df['order_date'].head())
print(df['client_singup_date'].dtype)
print(df['client_singup_date'].head())

datetime64[ns]
0   2024-09-13
1   2023-01-27
2   2023-03-19
3   2024-09-06
4   2024-01-15
Name: order_date, dtype: datetime64[ns]
datetime64[ns]
0   2022-04-05
1   2021-06-25
2   2024-11-02
3   2022-12-26
4   2024-12-26
Name: client_singup_date, dtype: datetime64[ns]


In [26]:
## 5. Validate

assert df['order_id'].is_unique, "order_id must be unique"
assert pd.api.types.is_datetime64_any_dtype(df['client_singup_date']), "client_singup_date must be datetime"
assert (df['quantity']>0).all(), "quantity must be positive"
assert df['discount_pct'].dropna().between(0,1).all(), "discount must be 0-1"
assert (df['country'].notna() & (df['country'] != '0')).all(), "no '0' or null in country"
assert (df['client_city'].notna() & (df['client_city'] != '0')).all(), "no '0' or null in client_city"
assert (df['product_category'].notna() & (df['product_category'] != '0')).all(), "no '0' or null in product_category"
assert (df['client_segment'] != '0').all(), "no '0' placeholders in segments"
assert (df['unit_price'] > 0).all(), "unit price must be positive"
assert pd.api.types.is_datetime64_any_dtype(df['order_date']), "order_date must be datetime"
assert pd.api.types.is_datetime64_any_dtype(df['client_singup_date']), "client_singup_date must be datetime"

print(" All validation checks passed -", len(df), "clean rows")

 All validation checks passed - 2497 clean rows


In [27]:
## 6. Build star schema

# ==== dim_product — one row per product.
# Resolve correlated supplier_name + product_active as a PAIR (most-common combo)
# so we never fabricate a (supplier, active) pairing that never occurred in the
# source; category/subcategory by modal consensus. Built AFTER cleaning + quarantine
# so the inputs are final (single source of truth for the product dimension).
modal = lambda s: (s.dropna().mode().iloc[0] if len(s.dropna().mode()) else np.nan)

def resolve_product(grp):
    combo = grp.groupby(['supplier_name', 'product_active']).size().idxmax()
    return pd.Series({
        'product_category':    modal(grp['product_category']),
        'product_subcategory': modal(grp['product_subcategory']),
        'supplier_name':       combo[0],
        'product_active':      combo[1],
    })

dim_product = (
    df.groupby('product_name')
      .apply(resolve_product, include_groups=False)
      .reset_index()
)
dim_product['product_key'] = dim_product.index + 1
print('dim_product rows:', len(dim_product))

# ==== dim_channel — one row per sales channel
dim_channel = df[['sales_channel']].drop_duplicates().reset_index(drop=True)
dim_channel['channel_key'] = dim_channel.index + 1
print('dim_channel rows:', len(dim_channel))

# ==== dim_date — full contiguous calendar from earliest to latest order
rng = pd.date_range(df['order_date'].min(), df['order_date'].max(), freq='D')
dim_date = pd.DataFrame({'order_date': rng})
dim_date['date_key']    = range(1, len(dim_date) + 1)
dim_date['year']        = dim_date['order_date'].dt.year
dim_date['quarter']     = dim_date['order_date'].dt.quarter
dim_date['month']       = dim_date['order_date'].dt.month
dim_date['month_name']  = dim_date['order_date'].dt.month_name()
dim_date['day_name']    = dim_date['order_date'].dt.day_name()
print('dim_date rows:', len(dim_date))

# ==== dim_customer — one row per customer, preferring real values over 'Unknown'
def first_real(s):
    s = s.replace('Unknown', np.nan).dropna()
    return s.iloc[0] if len(s) else 'Unknown'

dim_customer = (
    df.groupby('customer_name', as_index=False)
      .agg(
          client_segment     = ('client_segment', first_real),
          client_city        = ('client_city', first_real),
          country            = ('country', first_real),
          client_singup_date = ('client_singup_date', 'min')
      )
)

dim_customer['client_singup_date'] = dim_customer['client_singup_date'].dt.normalize()
dim_customer['customer_key'] = range(1, len(dim_customer) + 1)
print('dim_customer rows:', len(dim_customer))

# sanity-check the resolved product conflict actually stuck (expect Thomas Inc / False / Frozen)
print(dim_product[dim_product['product_name'] == 'Ice Cream Product 22']
      [['product_name', 'product_category', 'supplier_name', 'product_active']])
dim_customer.head()

dim_product rows: 79
dim_channel rows: 3
dim_date rows: 731


/tmp/ipykernel_2529/3107368377.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace('Unknown', np.nan).dropna()


dim_customer rows: 120
            product_name product_category supplier_name  product_active
31  Ice Cream Product 22           Frozen    Thomas Inc           False


,customer_name,client_segment,client_city,country,client_singup_date,customer_key
0,Acevedo-Lee,Distributor,Robertsmouth,Italy,2023-08-18,1
1,"Acosta, Welch and Hall",Hospitality,Reedberg,Portugal,2021-09-01,2
2,"Adams, Smith and Simpson",Distributor,South Matthewborough,Portugal,2024-04-24,3
3,Alexander PLC,Hospitality,North Crystalville,Spain,2026-03-11,4
4,Allen PLC,Retail,Davidstad,Italy,2024-02-03,5


In [28]:
# Sales the transaction
fact_sales = df.copy()

# join each table to bring in keys
fact_sales = fact_sales.merge(dim_customer[['customer_name', 'customer_key']], on='customer_name', how='left')
fact_sales = fact_sales.merge(dim_product[['product_name', 'product_key']], on='product_name', how='left')
fact_sales = fact_sales.merge(dim_channel[['sales_channel', 'channel_key']], on='sales_channel', how='left')
fact_sales = fact_sales.merge(dim_date[['order_date', 'date_key']], on='order_date', how='left')

#compute the measures
fact_sales['revenue'] = (fact_sales['quantity'] * fact_sales['unit_price'] * (1 - fact_sales['discount_pct'].fillna(0))).round(2)
fact_sales['cost'] = (fact_sales['quantity'] * fact_sales['unit_cost']).round(2)
fact_sales['profit'] = (fact_sales['revenue'] - fact_sales['cost']).round(2)

#order_id + keys+transactions measures
fact_cols = ['order_id', 'order_date', 'date_key', 'customer_key', 'product_key', 'channel_key', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'revenue', 'cost', 'profit']
fact_sales = fact_sales[fact_cols]

# fact <-> dim key integrity (every fact key must exist in its dimension)
key_cols = ['date_key','customer_key','product_key','channel_key']
assert fact_sales[key_cols].notna().all().all(), "null keys after merge!"
assert fact_sales['product_key'].isin(dim_product['product_key']).all(), "orphan product_key"

print('fact_sales rows:', len(fact_sales))
fact_sales

fact_sales rows: 2497


,order_id,order_date,date_key,customer_key,product_key,channel_key,quantity,unit_price,unit_cost,discount_pct,revenue,cost,profit
0,O00001,2024-09-13,622,66,52,1,33,57.98,12.48,0.06,1798.54,411.84,1386.70
1,O00002,2023-01-27,27,81,62,2,21,72.42,12.20,0.04,1459.99,256.20,1203.79
2,O00003,2023-03-19,78,28,54,1,49,30.96,7.57,0.25,1137.78,370.93,766.85
3,O00004,2024-09-06,615,21,28,3,90,58.39,38.22,0.30,3678.57,3439.80,238.77
4,O00005,2024-01-15,380,43,19,3,6,27.35,35.41,0.15,139.49,212.46,-72.97
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2492,O02496,2024-06-01,518,100,28,2,20,69.60,38.22,0.24,1057.92,764.40,293.52
2493,O02497,2023-03-25,84,106,39,3,52,32.61,8.88,0.12,1492.23,461.76,1030.47
2494,O02498,2023-11-18,322,61,32,3,62,71.24,9.69,0.23,3401.00,600.78,2800.22
2495,O02499,2023-02-01,32,65,70,2,56,13.25,39.97,0.17,615.86,2238.32,-1622.46


In [29]:
## 7. Export.
#export star-schema tables + quarantined rows for power bi
out = "/content/drive/MyDrive/Exceltis/"

fact_sales.to_csv(out + 'tbl_sales.csv', index=False)
dim_customer.to_csv(out + 'tbl_customer.csv', index=False)
dim_product.to_csv(out + 'tbl_product.csv', index=False)
dim_channel.to_csv(out + 'tbl_channel.csv', index=False)
dim_date.to_csv(out + 'tbl_date.csv', index=False)

# rejects = exactly the rows dropped in the quarantine cell (no re-read)
rejects.to_csv(out + 'tbl_rejects_quarantined.csv', index=False)

print(len(rejects), "quarantined row(s)")
print('all exported to Drive')


3 quarantined row(s)
all exported to Drive


In [30]:
print('df rows:', len(df))
print('unique customers:',df['customer_name'].nunique())
print('dim_customer rows:', len(dim_customer))
print('columns:', df.columns.tolist())

df rows: 2497
unique customers: 120
dim_customer rows: 120
columns: ['order_id', 'order_date', 'customer_name', 'client_segment', 'client_city', 'country', 'client_singup_date', 'product_name', 'quantity', 'unit_price', 'unit_cost', 'discount_pct', 'product_category', 'product_subcategory', 'supplier_name', 'product_active', 'sales_channel', 'sales_rep', 'product_family']


In [31]:
import os
print(os.listdir("/content/drive/MyDrive/Exceltis/"))

['Exeltis_Data_Engineer_Business_Case_Raw_Data.csv', 'tbl_customer.csv', 'tbl_channel.csv', 'tbl_product.csv', 'tbl_sales.csv', 'tbl_rejects_quarantined.csv', 'tbl_date.csv']
